# 🚀 AI Hub Google Colab GPU Bridge
### 100% Free 16GB Dedicated T4 GPU for True Spatio-Temporal Video Diffusion & SDXL

This notebook connects your free Google Colab **NVIDIA Tesla T4 (15.8 GB VRAM)** directly to AI Hub.

### Instructions:
1. In Google Colab menu: **Runtime -> Change runtime type -> T4 GPU**
2. Run the single cell below (`Shift + Enter`)
3. Copy the generated `https://xxxx.trycloudflare.com` URL
4. Paste it into AI Hub **Tab 3 -> Google Colab GPU Bridge** and click **Connect**!

In [ ]:
# 1. Install dependencies (Diffusers, FastAPI, Cloudflare)
!pip install -q fastapi uvicorn diffusers transformers accelerate torch pyngrok pillow imageio[ffmpeg] torchvision sentencepiece
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Dependencies and Cloudflare Tunnel installed successfully!")

# 2. Build the GPU Microservice
server_code = """
import os, io, time, base64
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel
import torch
from PIL import Image
import imageio
import numpy as np

app = FastAPI(title="AI Hub Colab GPU Bridge")

VIDEO_PIPE = None
IMAGE_PIPE = None

def get_video_pipeline():
    global VIDEO_PIPE
    if VIDEO_PIPE is None:
        from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
        print("⚡ Loading damo-vilab/text-to-video-ms-1.7b onto T4 GPU...")
        pipe = DiffusionPipeline.from_pretrained(
            "damo-vilab/text-to-video-ms-1.7b", 
            torch_dtype=torch.float16, 
            variant="fp16"
        )
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        pipe.enable_model_cpu_offload()
        VIDEO_PIPE = pipe
    return VIDEO_PIPE

def get_image_pipeline():
    global IMAGE_PIPE
    if IMAGE_PIPE is None:
        from diffusers import AutoPipelineForText2Image
        print("⚡ Loading stabilityai/sdxl-turbo onto T4 GPU...")
        pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sdxl-turbo",
            torch_dtype=torch.float16,
            variant="fp16"
        )
        pipe.to("cuda")
        IMAGE_PIPE = pipe
    return IMAGE_PIPE

@app.get("/health")
def health():
    gpu_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_available else "CPU (Warning: No GPU)"
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1) if gpu_available else 0
    return {
        "status": "healthy",
        "gpu": gpu_name,
        "vram_gb": vram_gb,
        "cuda_version": torch.version.cuda if gpu_available else None
    }

class ImageGenRequest(BaseModel):
    prompt: str
    width: int = 1024
    height: int = 1024
    num_inference_steps: int = 2

@app.post("/generate-image")
def generate_image(req: ImageGenRequest):
    try:
        pipe = get_image_pipeline()
        img = pipe(prompt=req.prompt, num_inference_steps=max(1, min(req.num_inference_steps, 4)), guidance_scale=0.0).images[0]
        if (img.width, img.height) != (req.width, req.height):
            img = img.resize((req.width, req.height), Image.Resampling.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format="WEBP", quality=92)
        return Response(content=buf.getvalue(), media_type="image/webp")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

class VideoGenRequest(BaseModel):
    prompt: str
    width: int = 512
    height: int = 512
    duration: int = 4
    fps: int = 16

@app.post("/generate-video")
def generate_video(req: VideoGenRequest):
    try:
        pipe = get_video_pipeline()
        num_frames = max(16, min(int(req.duration * 6), 32))
        video_frames = pipe(req.prompt, num_frames=num_frames, height=req.height, width=req.width, num_inference_steps=25).frames[0]
        buf = io.BytesIO()
        if isinstance(video_frames, np.ndarray):
            if video_frames.dtype != np.uint8:
                video_frames = (video_frames * 255).astype(np.uint8)
        writer = imageio.get_writer(buf, format="mp4", fps=req.fps, codec="libx264")
        for frame in video_frames:
            if not isinstance(frame, np.ndarray):
                frame = np.array(frame)
            if frame.dtype != np.uint8:
                frame = (frame * 255).astype(np.uint8)
            writer.append_data(frame)
        writer.close()
        return Response(content=buf.getvalue(), media_type="video/mp4")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
"""

with open("colab_bridge_server.py", "w") as f:
    f.write(server_code)

# 3. Launch Server & Start Cloudflare Tunnel
import subprocess, time, re

proc = subprocess.Popen(["uvicorn", "colab_bridge_server:app", "--port", "8000"])
time.sleep(2)

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print("🚀 Starting Cloudflare Tunnel... Generating public endpoint...")
for _ in range(35):
    line = tunnel.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "="*60)
            print("🎉 YOUR COLAB GPU BRIDGE URL IS READY:")
            print(f"👉  {url}  👈")
            print("="*60)
            print("Copy this link and paste it into AI Hub Tab 3 -> Google Colab GPU Bridge!")
            break
    time.sleep(0.5)
